# 🛡️ INTERCEPT CDSS v2.2 — Teste BioBERTpt
> Motor determinístico + NER clínico em Português
>
> Regras: ANVISA · SBC 2023 · SBD 2024 · SBPT · AHA · FDA

In [ ]:
# ── CÉLULA 1: Dependências ─────────────────────────────────────────────────
%pip install -q transformers ipywidgets
# torch (necessário para o BioBERTpt):
%pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# ── CÉLULA 2: Carrega o motor INTERCEPT ────────────────────────────────────
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('/content/Nesis.AI'):
        subprocess.run(['git', 'clone',
            'https://github.com/gabrielbringel/Nesis.AI.git',
            '/content/Nesis.AI'], check=True)
    sys.path.insert(0, '/content/Nesis.AI')
else:
    sys.path.insert(0, os.path.abspath(''))

from intercept_engine import InterceptCDSS, display_resultado

print('✅ Motor INTERCEPT CDSS v2.2 carregado!')
print(f'   📂 Path: {sys.path[0]}')
print('   • 30 regras determinísticas (ANVISA/SBC/FDA)')
print('   • 100+ fármacos | fuzzy match ativo')

In [ ]:
# ── CÉLULA 3: Inicializa BioBERTpt (NER) ──────────────────────────────────
# O BioBERTpt extrai entidades médicas do texto livre.
# Se falhar (DLL/torch), o motor usa dicionário+fuzzy automaticamente.

ner_pipeline = None

try:
    from transformers import pipeline
    import torch
    device = 0 if torch.cuda.is_available() else -1
    print(f'🖥️  Dispositivo: {"GPU" if device == 0 else "CPU"}')
    print('⏳ Carregando pucpr/clinicalnerpt-medical...')
    ner_pipeline = pipeline(
        'ner',
        model='pucpr/clinicalnerpt-medical',
        aggregation_strategy='simple',
        device=device
    )
    print('✅ BioBERTpt NER carregado com sucesso!')
    print('   → Entidades extraídas terão fonte_extracao: "biobertpt" e score NER.')
except Exception as e:
    print(f'⚠️  BioBERTpt indisponível: {type(e).__name__}: {e}')
    print('   → Modo fallback: dicionário+fuzzy (fonte_extracao: "dicionario")')

cdss = InterceptCDSS(ner_pipeline=ner_pipeline)
modo = 'BioBERTpt NER + dicionário' if ner_pipeline else 'Dicionário+Fuzzy (fallback)'
print(f'\n🛡️ INTERCEPT pronto | Modo de extração: {modo}')

In [ ]:
# ── CÉLULA 4: Casos clínicos + Painel Interativo ───────────────────────────
import ipywidgets as widgets
from IPython.display import display as ipy_display, clear_output

CASOS = {
    "🔴 Caso 1 — IECA + Espironolactona (Hipercalemia)": """S: Paciente masculino, 65 anos, UBS com queixa de tosse e dores no corpo.
Relata HAS de longa data e DM2. Nega alergias.
Em uso contínuo: enalapriu 20mg 1x/dia, aas infantil 100mg, metformina 850mg.
O: PA 150x90. FC 82. Sat 97%. Corado, hidratado.
A: Mialgia. PA descompensada. Tosse possivelmente por IECA.
P: Médico considera prescrever Espironolactona 25mg e Ibuprofeno 400mg.""",

    "🔴 Caso 2 — Asma + Propranolol (Broncoespasmo)": """S: Paciente feminina, 42 anos. Asma brônquica moderada.
Em uso: Salbutamol spray e Beclometasona inalatória.
UPA com palpitações e PA elevada. Nega tabagismo.
O: PA 165x100. FC 112. Ausculta com sibilância.
A: Taquicardia sinusal + HAS estágio 2 em asmática.
P: Médico considera iniciar Propranolol 40mg 2x/dia.""",

    "🔴 Caso 3 — DRC + Metformina + Varfarina + AINE": """S: Paciente masculino, 72 anos. Diabético tipo 2 há 15 anos.
Insuficiência renal crônica (creatinina 2,8 mg/dL, TFG 28 mL/min).
Em uso: varfarina 5mg (FA) e atorvastatina 40mg. Dor articular (gota?).
O: PA 138x88. FC 74. INR 2,8. Ureia 98. Creatinina 2,8.
A: Artralgia gotosa. DRC G4. Anticoagulação terapêutica.
P: Médico considera metformina 850mg + ibuprofeno 600mg.""",

    "🔴 Caso 4 — Nitrato + Sildenafil (Hipotensão Fatal)": """S: Paciente masculino, 60 anos. Angina estável.
Em uso: mononitrato de isossorbida 20mg 2x/dia e atenolol 50mg.
Refere disfunção erétil e solicita tratamento.
O: PA 138x84. FC 72. ECG sem alterações agudas.
A: Angina estável controlada. Disfunção erétil.
P: Médico considera prescrever sildenafil 50mg.""",

    "🔴 Caso 5 — Gestante + IECA (Teratogenicidade)": """S: Paciente feminina, 28 anos, gestante 18 semanas.
HAS crônica antes da gestação. Em uso de enalapril 10mg.
Pré-natal regular sem outras intercorrências.
O: PA 148x96. FC 88. BCF fetal 148bpm.
A: HAS crônica na gestação com medicação de risco teratogênico.
P: Avaliar manutenção ou troca do anti-hipertensivo.""",

    "🟡 Caso 6 — Estatina + Genfibrozila (Miopatia)": """S: Paciente masculino, 55 anos. Dislipidemia mista.
Em uso: sinvastatina 40mg há 2 anos. Sem comorbidades.
Labs: Colesterol total 280, TG 450, HDL 32, CPK normal.
O: PA 128x82. FC 76. Sem queixas musculares.
A: Dislipidemia mista não controlada com monoterapia.
P: Médico considera adicionar Genfibrozila 600mg 2x/dia.""",

    "🟡 Caso 7 — Opioide + Benzodiazepínico (Depressão Respiratória)": """S: Paciente masculino, 48 anos, pós-operatório ortopédico.
Em uso crônico: clonazepam 2mg para ansiedade.
Dor intensa no local cirúrgico (EVA 8/10).
O: PA 125x78. FC 80. Sat O2 97%.
A: Dor aguda pós-operatória em usuário crônico de benzodiazepínico.
P: Médico considera prescrever morfina 10mg.""",

    "🟢 Caso 8 — Sem Interações (Controle Negativo)": """S: Paciente feminina, 45 anos. HAS controlada. Sem comorbidades.
Em uso: losartana 50mg e anlodipino 5mg há 1 ano. Boa tolerância.
O: PA 122x78. FC 68. Labs normais. K+ 4,1.
A: HAS controlada com esquema duplo.
P: Manter medicações. Médico considera omeprazol 20mg profilático.""",

    "✏️ Caso Personalizado": """Descreva aqui o caso clínico que deseja testar.
Inclua: idade, sexo, diagnósticos, medicamentos em uso,
exames relevantes (creatinina, K+, INR, etc.) e
o(s) medicamento(s) que o médico pretende prescrever."""
}

# ── Widgets ────────────────────────────────────────────────────────────────
header = widgets.HTML("""
<div style='background:#0f1117;padding:14px 18px;border-radius:10px;
            border-left:4px solid #e94560;margin-bottom:10px'>
  <h3 style='color:#e94560;margin:0;font-family:monospace;letter-spacing:1px'>
      🛡️ INTERCEPT CDSS — Teste BioBERTpt</h3>
  <p style='color:#a0aec0;margin:6px 0 0 0;font-size:12px'>
      Selecione um caso ou escreva um prontuário livre e clique
      <b style='color:#e94560'>Analisar</b>. O resultado mostra quais
      entidades o BioBERTpt reconheceu e quais alertas foram disparados.
  </p>
</div>
""")

dropdown = widgets.Dropdown(
    options=list(CASOS.keys()),
    description='Caso:',
    style={'description_width': '40px'},
    layout=widgets.Layout(width='99%')
)

txt = widgets.Textarea(
    value=CASOS[dropdown.value].strip(),
    placeholder='Cole o prontuário aqui...',
    layout=widgets.Layout(width='99%', height='220px')
)

btn = widgets.Button(
    description='🔍 Analisar Prontuário',
    button_style='danger',
    layout=widgets.Layout(width='220px', height='40px')
)

btn_clear = widgets.Button(
    description='🗑️ Limpar',
    button_style='warning',
    layout=widgets.Layout(width='100px', height='40px')
)

out = widgets.Output()

def on_dropdown(change):
    if change['name'] == 'value':
        txt.value = CASOS[change['new']].strip()
        with out:
            clear_output()

def on_analisar(b):
    texto = txt.value.strip()
    if not texto or texto.startswith('Descreva aqui'):
        with out:
            clear_output()
            print('⚠️ Insira o texto do prontuário antes de analisar.')
        return
    btn.disabled = True
    btn.description = '⏳ Analisando...'
    with out:
        clear_output(wait=True)
        try:
            resultado = cdss.analisar_prontuario(texto)
            display_resultado(resultado)
        except Exception as e:
            print(f'❌ Erro: {e}')
    btn.disabled = False
    btn.description = '🔍 Analisar Prontuário'

def on_limpar(b):
    txt.value = ''
    with out:
        clear_output()

dropdown.observe(on_dropdown)
btn.on_click(on_analisar)
btn_clear.on_click(on_limpar)

botoes = widgets.HBox([btn, btn_clear], layout=widgets.Layout(gap='8px', margin='6px 0'))
ipy_display(widgets.VBox([header, dropdown, txt, botoes, out]))